1. Join all static and temporal features
2. Train test validation split
3. Scale numerical features

Model-fitting period: January 2021–March 2024, 114,842 rows

Validation period: April 2024–March 2025, 35,158 rows

Temporal join on Month and Year
Static join on location

In [ ]:
import pandas as pd
import numpy as np
import scipy as scp
import math
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib
import os.path as osp
import sys
# -- Paths --
ROOT = '/Users/keltonn/Desktop/Projects/housing_rent_price/final_report'
sys.path.append(ROOT)
from data.feature_config_static import (
    ID_COL,
    HOT_COL,
    LABEL,
    NUMERICAL_COL,
    CATEGORICAL_COL
    )
from data.feature_config_temporal import (
    STOCK_FEATURE,
    COE_FEATURE,
    MACRO_FEATURE
    )

# Static Features
1. Train Validation Test split
2. Scaling on Train for numerical columns
3. Combine into single dataframe and save

In [27]:
# Static Features
# 1. Train/Test/Validation Split
static_train = pd.read_csv(f'{ROOT}/data/features/cleaned/static_features/one_hot/train_one_hot.csv')
ym = static_train['RENT_APPROVAL_YEAR'] * 100 + static_train['RENT_APPROVAL_MONTH']
static_train['sample_type'] = np.where(ym <= 202403, 'train', 'val')

static_test = pd.read_csv(f'{ROOT}/data/features/cleaned/static_features/one_hot/test_one_hot.csv')
static_test['sample_type'] = 'test'

In [28]:
SCALER_PATH = f'{ROOT}/data/features/final/scalers/static_numerical_scalers.pkl'
SCALE_FT = NUMERICAL_COL

try:
    scaler = joblib.load(SCALER_PATH)
except:
    print("No scaler trained. Training new scaler")
    scaler = StandardScaler()
    train_set = static_train[static_train['sample_type'] == 'train'].reset_index(drop = True)
    scaler.fit(train_set[SCALE_FT])
    joblib.dump(scaler, SCALER_PATH)

scaler = joblib.load(SCALER_PATH)
train_df = static_train.copy(deep = True)
test_df = static_test.copy(deep = True)
train_df[SCALE_FT] = scaler.transform(train_df[SCALE_FT])
train_df = train_df[
    ['sample_type', LABEL] + ID_COL + HOT_COL + CATEGORICAL_COL + NUMERICAL_COL
]
test_df[SCALE_FT] = scaler.transform(test_df[SCALE_FT])
test_df = test_df[
    ['sample_type'] + ID_COL + HOT_COL + CATEGORICAL_COL + NUMERICAL_COL
]
train_df

No scaler trained. Training new scaler


,sample_type,MONTHLY_RENT,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,LEASE_COMMENCE_DATE,TOWN,BLOCK,STREET,POSTAL_CODE,LATITUDE,...,jc_arts_entry_min,jc_arts_entry_max,jc_science_entry_min,jc_science_entry_max,mrt_traffic_tap_out_amount_norm,mrt_traffic_total_tap_norm,mrt_traffic_weekday_tap_norm,mrt_traffic_weekend_tap_norm,mrt_traffic_am_peak_norm,mrt_traffic_pm_peak_norm
0,train,1800,2021,1,2012,punggol,604a,punggol road,821604,1.404310,...,0.709636,0.512897,1.304346,0.669106,1.063248,1.130934,1.057059,1.361935,0.681458,1.030185
1,train,2000,2021,1,2001,sengkang,290c,compassvale crescent,543290,1.397018,...,0.709636,0.188720,-0.019002,0.168584,1.255873,1.255145,1.263346,1.204674,1.123844,1.256461
2,train,2000,2021,1,1976,queenstown,5,ghim moh road,270005,1.312693,...,-0.198541,0.026632,-0.349839,-0.165097,0.185327,0.230431,0.338227,-0.130223,0.467895,0.628689
3,train,1400,2021,1,2010,sengkang,436b,fernvale road,792436,1.392878,...,0.709636,0.188720,-0.019002,0.168584,1.255873,1.255145,1.263346,1.204674,1.123844,1.256461
4,train,1800,2021,1,2001,sembawang,467b,admiralty drive,752467,1.451141,...,0.104184,1.809602,1.966020,1.670150,0.897858,0.918661,0.921912,0.893946,1.109589,0.839903
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149994,val,2500,2025,3,1985,bukit batok,220,bukit batok east avenue 3,650220,1.347456,...,1.617812,1.809602,1.966020,2.003831,0.350072,0.287525,0.301814,0.242249,0.298617,0.375817
149995,val,5300,2025,3,2020,queenstown,79,dawson road,141079,1.294591,...,-0.198541,0.026632,-0.349839,-0.165097,-0.493499,-0.481125,-0.474993,-0.478248,-0.404190,-0.418597
149996,val,2650,2025,3,1971,toa payoh,17,lorong 7 toa payoh,310017,1.336427,...,0.104184,0.188720,-0.019002,0.001743,0.706263,0.683482,0.681397,0.683128,0.621542,0.619828
149997,val,3300,2025,3,2001,sembawang,360a,admiralty drive,751360,1.449203,...,0.104184,1.809602,1.966020,1.670150,0.897858,0.918661,0.921912,0.893946,1.109589,0.839903


In [29]:
FINAL_SCALED_PATH = f'{ROOT}/data/features/final'
train_df.to_csv(f'{FINAL_SCALED_PATH}/static_train_scaled.csv', index = False)
test_df.to_csv(f'{FINAL_SCALED_PATH}/static_test_scaled.csv', index = False)

# Temporal Features
1. Train Validation Test split
2. Scaling on Train for numerical columns
3. Combine into single dataframe and save

In [38]:
# temporal features
temp_df = pd.read_csv(f'{ROOT}/data/features/cleaned/temporal_features/selected_temporal_features.csv')
ym = temp_df['RENT_APPROVAL_YEAR'] * 100 + temp_df['RENT_APPROVAL_MONTH']

temp_df['sample_type'] = np.select(
    [ym <= 202403, ym <= 202503],
    ['train', 'val'],
    default='test',
)
temp_df[
    ['sample_type'] + temp_df.columns.tolist()
].to_csv(f'{ROOT}/data/features/final/temporal_raw.csv',index = False)
temp_df

,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,stk_ascendas_reit_log_price_mean,stk_ascendas_reit_mom_return,stk_ascendas_reit_return_volatility,stk_boc_aviation_log_price_mean,stk_boc_aviation_mom_return,stk_boc_aviation_return_volatility,stk_capitaland_ascott_trust_log_price_mean,stk_capitaland_ascott_trust_mom_return,...,coe_b_bids_received,coe_b_quota,coe_b_quota_premium,coe_c_bids_received,coe_c_quota,coe_c_quota_premium,coe_e_bids_received,coe_e_quota,coe_e_quota_premium,sample_type
0,2021,1,0.806770,NaN,0.012960,3.925360,NaN,0.024428,-0.267930,NaN,...,1367.0,1020.5,49550.5,675.0,376.5,38449.5,731.5,466.5,50694.5,train
1,2021,2,0.810418,0.003648,0.010391,3.962258,0.036898,0.024973,-0.313300,-0.045370,...,1137.5,942.5,46396.0,467.0,267.5,42945.5,469.0,302.0,48642.0,train
2,2021,3,0.786484,-0.023934,0.012611,4.087016,0.124758,0.023987,-0.258145,0.055155,...,1171.5,944.0,46001.0,493.5,273.5,38551.0,440.0,297.0,47403.5,train
3,2021,4,0.824610,0.038125,0.006253,4.034332,-0.052684,0.020803,-0.249994,0.008150,...,1554.5,957.0,56749.5,464.5,262.5,40067.5,634.0,299.5,57150.0,train
4,2021,5,0.792135,-0.032474,0.012063,4.001513,-0.032818,0.016937,-0.306375,-0.056381,...,1166.5,818.0,59045.0,288.0,160.5,42351.0,393.5,267.0,62501.0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,2026,3,0.908096,-0.063457,0.009846,4.335577,-0.094832,0.023054,-0.126413,-0.066780,...,1223.0,813.5,114785.0,463.0,290.0,77000.0,468.0,245.5,116504.5,test
63,2026,4,0.917551,0.009455,0.009791,4.374565,0.038988,0.018064,-0.128853,-0.002441,...,1300.0,811.0,121000.5,516.5,292.5,81751.0,484.5,243.0,123001.5,test
64,2026,5,0.897899,-0.019652,0.006192,4.334378,-0.040187,0.013330,-0.142752,-0.013899,...,1400.5,876.0,127868.5,489.5,292.5,89851.0,457.5,255.0,128850.0,test
65,2026,6,0.905678,0.007779,0.009558,4.337397,0.003019,0.015430,-0.137612,0.005140,...,1251.0,886.0,125245.5,421.5,292.0,93500.5,492.0,261.5,129001.0,test


In [39]:
SCALER_PATH = f'{ROOT}/data/features/final/scalers/temporal_numerical_scalers.pkl'
SCALE_FT =     STOCK_FEATURE + COE_FEATURE + MACRO_FEATURE

try:
    scaler = joblib.load(SCALER_PATH)
except:
    print("No scaler trained. Training new scaler")
    scaler = StandardScaler()
    train_set = temp_df[temp_df['sample_type'] == 'train'].reset_index(drop = True)
    scaler.fit(train_set[SCALE_FT])
    joblib.dump(scaler, SCALER_PATH)

scaler = joblib.load(SCALER_PATH)
train_df = temp_df.copy(deep = True)
train_df[SCALE_FT] = scaler.transform(train_df[SCALE_FT])
train_df = train_df[
    ['sample_type'] + temp_df.columns.tolist()
]
train_df

,sample_type,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,stk_ascendas_reit_log_price_mean,stk_ascendas_reit_mom_return,stk_ascendas_reit_return_volatility,stk_boc_aviation_log_price_mean,stk_boc_aviation_mom_return,stk_boc_aviation_return_volatility,stk_capitaland_ascott_trust_log_price_mean,...,coe_b_bids_received,coe_b_quota,coe_b_quota_premium,coe_c_bids_received,coe_c_quota,coe_c_quota_premium,coe_e_bids_received,coe_e_quota,coe_e_quota_premium,sample_type
0,train,2021,1,-0.290287,NaN,0.967794,-0.115539,NaN,0.615515,-0.619635,...,2.026297,2.394230,-1.721730,3.389484,3.361603,-1.266426,3.904067,4.166700,-1.735850,train
1,train,2021,2,-0.189146,0.075717,0.029754,0.457438,0.515826,0.695686,-1.388492,...,1.099719,1.941672,-1.841444,1.753140,1.876447,-1.010323,1.419043,1.718316,-1.811951,train
2,train,2021,3,-0.852714,-0.808712,0.840058,2.394769,1.730575,0.550679,-0.453810,...,1.236990,1.950375,-1.856435,1.961616,1.958199,-1.260644,1.144507,1.643897,-1.857871,train
3,train,2021,4,0.204321,1.181262,-1.481146,1.576657,-0.722726,0.082721,-0.315688,...,2.783305,2.025801,-1.448525,1.733472,1.808320,-1.174261,2.981058,1.681106,-1.496498,train
4,train,2021,5,-0.696039,-1.082576,0.640172,1.067028,-0.448069,-0.485483,-1.271140,...,1.216803,1.219319,-1.361410,0.344939,0.418541,-1.044187,0.704303,1.197383,-1.298098,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,test,2026,3,2.518992,-2.076040,-0.169344,6.254604,-1.305466,0.413652,1.778576,...,1.444915,1.193210,0.753944,1.721671,2.183016,0.929496,1.409576,0.877381,0.704201,test
63,test,2026,4,2.781141,0.261932,-0.189260,6.860041,0.544727,-0.319784,1.737216,...,1.755792,1.178705,0.989825,2.142558,2.217079,1.200124,1.565778,0.840171,0.945091,test
64,test,2026,5,2.236272,-0.671430,-1.503327,6.235994,-0.549941,-1.015607,1.501683,...,2.161549,1.555837,1.250468,1.930148,2.217079,1.661518,1.310175,1.018777,1.161937,test
65,test,2026,6,2.451951,0.208186,-0.274323,6.282873,0.047417,-0.706930,1.588781,...,1.557961,1.613857,1.150924,1.395189,2.210267,1.869401,1.636778,1.115522,1.167536,test


In [40]:
FINAL_SCALED_PATH = f'{ROOT}/data/features/final'
train_df.to_csv(f'{FINAL_SCALED_PATH}/temporal_scaled.csv', index = False)